# Gradient Accumulation Smoke Test

Bu notebook yalnizca tek bir accumulation kosusunun calisabilirligini ve artifact olusumunu dogrular.

In [1]:
from pathlib import Path
import os
import sys
import json
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset
from torchvision import models

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for parent in [current, *current.parents]:
        if (parent / ".git").exists():
            return parent
    raise RuntimeError("Project root not found (no .git folder).")

PROJECT_ROOT = find_project_root()
GA_PATH = (PROJECT_ROOT / "gradient_accumulation").resolve()
if str(GA_PATH) not in sys.path:
    sys.path.insert(0, str(GA_PATH))

from functions.dataset import COVIDCXNetDataset, DataLoaderConfig, build_transforms, create_dataloader
from functions.train import TrainConfig, fit
import functions.train as train_module

expected_train_path = (GA_PATH / "functions" / "train.py").resolve()
actual_train_path = Path(train_module.__file__).resolve()
assert actual_train_path == expected_train_path, (
    f"Wrong train module imported. Expected: {expected_train_path}, got: {actual_train_path}"
)
print(f"[OK] Imported train module from: {actual_train_path}")

[OK] Imported train module from: /home/furkan/Projects/ML_Algorithms/gradient_accumulation/functions/train.py


In [2]:
PROFILE_NAME = "smoke"
CONFIG_PATH = PROJECT_ROOT / "batchsize" / "configs" / "gradient_accumulation_experiment.json"

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    CONFIG = json.load(f)

PROFILE = CONFIG["profiles"][PROFILE_NAME]
RUNS_ROOT = Path(CONFIG["runs_root"])
if not RUNS_ROOT.is_absolute():
    RUNS_ROOT = (PROJECT_ROOT / RUNS_ROOT).resolve()

def set_seed_like_previous_experiment(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = bool(deterministic)
    torch.backends.cudnn.benchmark = not bool(deterministic)

set_seed_like_previous_experiment(CONFIG["seed"], CONFIG["deterministic"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Profile: {PROFILE_NAME} | Device: {device}")

Profile: smoke | Device: cuda


In [3]:
train_transform = build_transforms(image_size=CONFIG["image_size"], augment=True)
eval_transform = build_transforms(image_size=CONFIG["image_size"], augment=False)

train_dataset = COVIDCXNetDataset(
    csv_file=CONFIG["data_csv_file"],
    root_dir=CONFIG["data_root_dir"],
    transform=train_transform,
    split="train",
)

val_dataset = None
for split_name in CONFIG["val_split_candidates"]:
    try:
        val_dataset = COVIDCXNetDataset(
            csv_file=CONFIG["data_csv_file"],
            root_dir=CONFIG["data_root_dir"],
            transform=eval_transform,
            split=split_name,
        )
        break
    except ValueError:
        continue

if val_dataset is None:
    raise ValueError("No validation split found in configured candidates.")

g = torch.Generator()
g.manual_seed(CONFIG["seed"])

train_n = min(int(PROFILE["train_samples"]), len(train_dataset))
val_n = min(int(PROFILE["val_samples"]), len(val_dataset))

train_indices = torch.randperm(len(train_dataset), generator=g)[:train_n].tolist()
val_indices = torch.randperm(len(val_dataset), generator=g)[:val_n].tolist()

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)

print(f"Train subset: {len(train_subset)}")
print(f"Val subset: {len(val_subset)}")

Train subset: 1024
Val subset: 512


In [4]:
def build_resnet50(num_classes: int) -> nn.Module:
    try:
        model = models.resnet50(weights="DEFAULT")
    except Exception:
        model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

micro = int(PROFILE["micro_batch_size"])
effective = int(PROFILE["effective_batch_sizes"][0])
assert effective % micro == 0, "Smoke config invalid: effective batch must be divisible by micro batch."
acc_steps = effective // micro

run_name = f"ga_resnet50_mb{micro}_ebs{effective}_acc{acc_steps}_{PROFILE_NAME}"

train_loader = create_dataloader(
    train_subset,
    DataLoaderConfig(batch_size=micro, shuffle=True, num_workers=CONFIG["num_workers"], drop_last=False),
    device=device,
)
val_loader = create_dataloader(
    val_subset,
    DataLoaderConfig(batch_size=micro, shuffle=False, num_workers=CONFIG["num_workers"], drop_last=False),
    device=device,
)

model = build_resnet50(CONFIG["num_classes"])
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=float(CONFIG["optimizer"]["lr"]))

train_cfg = TrainConfig(
    num_epochs=int(PROFILE["num_epochs"]),
    accumulation_steps=int(acc_steps),
    patience=1,
    run_name=run_name,
    output_dir=str(RUNS_ROOT),
    log_every_n_steps=5,
    system_log_interval=10,
    save_best=True,
)

result = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    config=train_cfg,
    logger=None,
    show_progress=True,
)

run_dir = Path(result.run_dir)
print(f"Run dir: {run_dir}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/furkan/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:04<00:00, 21.7MB/s]
/home/furkan/Projects/ML_Algorithms/gradient_accumulation/functions/train.py:318: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=amp_enabled)
[2026-02-27 23:27:52] Run 'ga_resnet50_mb32_ebs1024_acc32_smoke' started | epochs=1, accumulation_steps=32, amp=True
Epoch 1 [train]:   0%|          | 0/32 [00:00<?, ?it/s]/home/furkan/Projects/ML_Algorithms/gradient_accumulation/functions/train.py:170: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp_enabled):
[2026-02-27 23:27:59] Epoch 1/1 | train_loss=0.6901, train_acc=0.5254, val_loss=0.6596, val_acc=0.6895, epoch_time=4.68s, train_sps=219.01, 

Run dir: /home/furkan/Projects/ML_Algorithms/batchsize/runs/gradient_accumulation_compare/ga_resnet50_mb32_ebs1024_acc32_smoke


In [5]:
expected_files = [
    "step_metrics.csv",
    "epoch_metrics.csv",
    "system_metrics.csv",
    "run_summary.json",
    "run.log",
]

missing = [name for name in expected_files if not (run_dir / name).exists()]
if missing:
    print("FAIL")
    print(f"Missing artifacts: {missing}")
else:
    print("PASS")
    print(f"Artifacts complete in: {run_dir}")

PASS
Artifacts complete in: /home/furkan/Projects/ML_Algorithms/batchsize/runs/gradient_accumulation_compare/ga_resnet50_mb32_ebs1024_acc32_smoke
